In [1]:
from wsgiref import headers

import requests

cik = "0001318605"
headers = {'User-Agent': 'Marco Sison marcosison.1558@gmail.com'}
url = f"http://data.sec.gov/submissions/CIK{cik}.json"
response = requests.get(url, headers=headers)
data = response.json()
forms = data['filings']['recent']['form']
dates = data['filings']['recent']['filingDate']
accession_numbers = data['filings']['recent']['accessionNumber']
primary_documents = data['filings']['recent']['primaryDocument']
target_index = None
for i,form in enumerate(forms):
    if form == '10-K':
        print(f"Form: {form}, Filing Date: {dates[i]}, Accession Number: {accession_numbers[i]},Primary Document: {primary_documents[i]}")
        target_index = i
        break
accession_number = accession_numbers[target_index].replace("-", "")
doc_url = f"https://www.sec.gov/Archives/edgar/data/{cik}/{accession_number}/{primary_documents[target_index]}"
doc_response = requests.get(doc_url, headers=headers)
print(doc_response.text[:500])


Form: 10-K, Filing Date: 2026-01-29, Accession Number: 0001628280-26-003952,Primary Document: tsla-20251231.htm
<?xml version='1.0' encoding='ASCII'?>
<!--XBRL Document Created with the Workiva Platform-->
<!--Copyright 2026 Workiva-->
<!--r:e8b1bfdd-7bcd-4fd0-883e-f27688c85757,g:c1605f49-9f85-4158-99d4-5f5168bc8ff9,d:39f9a11fdbe840828cbaefa2e098ffa8-->
<html xmlns="http://www.w3.org/1999/xhtml" xmlns:ixt="http://www.xbrl.org/inlineXBRL/transformation/2020-02-12" xmlns:ixt-sec="http://www.sec.gov/inlineXBRL/transformation/2015-08-31" xmlns:ix="http://www.xbrl.org/2013/inlineXBRL" xmlns:xsi="http://www.w3.


In [2]:
def get_latest_10k(cik):
    headers = {'User-Agent': 'Marco Sison marcosison.1558@gmail.com'}
    url = f"http://data.sec.gov/submissions/CIK{cik}.json"
    response = requests.get(url, headers=headers)
    data = response.json()
    forms = data['filings']['recent']['form']
    dates = data['filings']['recent']['filingDate']
    accession_numbers = data['filings']['recent']['accessionNumber']
    primary_documents = data['filings']['recent']['primaryDocument']
    target_index = None
    for i,form in enumerate(forms):
        if form == '10-K':
            print(f"Form: {form}, Filing Date: {dates[i]}, Accession Number: {accession_numbers[i]},Primary Document: {primary_documents[i]}")
            target_index = i
            break
    accession_number = accession_numbers[target_index].replace("-", "")
    doc_url = f"https://www.sec.gov/Archives/edgar/data/{cik}/{accession_number}/{primary_documents[target_index]}"
    doc_response = requests.get(doc_url, headers=headers)
    return doc_response.text
print(get_latest_10k(cik)[:500])

Form: 10-K, Filing Date: 2026-01-29, Accession Number: 0001628280-26-003952,Primary Document: tsla-20251231.htm
<?xml version='1.0' encoding='ASCII'?>
<!--XBRL Document Created with the Workiva Platform-->
<!--Copyright 2026 Workiva-->
<!--r:e8b1bfdd-7bcd-4fd0-883e-f27688c85757,g:c1605f49-9f85-4158-99d4-5f5168bc8ff9,d:39f9a11fdbe840828cbaefa2e098ffa8-->
<html xmlns="http://www.w3.org/1999/xhtml" xmlns:ixt="http://www.xbrl.org/inlineXBRL/transformation/2020-02-12" xmlns:ixt-sec="http://www.sec.gov/inlineXBRL/transformation/2015-08-31" xmlns:ix="http://www.xbrl.org/2013/inlineXBRL" xmlns:xsi="http://www.w3.


In [3]:
from bs4 import BeautifulSoup
raw_html = get_latest_10k(cik)
def clean_html(raw_html):
    soup = BeautifulSoup(raw_html, 'html.parser')
    for tag in soup(['script', 'style']):
        tag.decompose()
    for tag in soup.find_all(style=lambda s: s and 'display:none' in s.replace(' ', '')):
        tag.decompose()
    clean_text = soup.get_text(separator=' ', strip=True)
    return clean_text
clean_text = clean_html(raw_html)
print(clean_text[:500])

Form: 10-K, Filing Date: 2026-01-29, Accession Number: 0001628280-26-003952,Primary Document: tsla-20251231.htm
tsla-20251231 UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 FORM 10-K (Mark One) x ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 For the fiscal year ended December 31 , 2025 OR o TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 For the transition period from _________ to _________ Commission File Number: 001-34756 Tesla, Inc. (Exact name of registrant as specified in its charter) Texas 91-219


In [4]:
import os
def save_filing_text(text, filepath):
    os.makedirs('data', exist_ok=True)
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(text)
save_filing_text(clean_text, 'data/tsla_10k_2025.txt')

In [5]:
def chunk_text(data, chunk_size=1000, overlap=200):
    chunks = []
    start = 0
    while start < len(data):
        end = start + chunk_size
        chunk = data[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks
chunks = chunk_text(clean_text)
print(len(chunks))
print(chunks[0][-250:])
print("---")
print(chunks[1][:250])

495
area code) Securities registered pursuant to Section 12(b) of the Act: Title of each class Trading Symbol(s) Name of each exchange on which registered Common stock TSLA The Nasdaq Global Select Market Securities registered pursuant to Section 12(g) o
---
on 12(b) of the Act: Title of each class Trading Symbol(s) Name of each exchange on which registered Common stock TSLA The Nasdaq Global Select Market Securities registered pursuant to Section 12(g) of the Act: None Indicate by check mark whether the


In [9]:
import voyageai
from dotenv import load_dotenv

load_dotenv()
vo = voyageai.Client()
result =vo.embed(["Tesla's revenue grew significantly in 2025."], model="voyage-4")
print(len(result.embeddings[0]))
print(result.embeddings[0][:10])

1024
[-0.025886787101626396, -0.027640439569950104, -0.05184657499194145, 0.011342007666826248, -0.024197574704885483, 0.027991119772195816, -0.0301645640283823, -0.05073791742324829, 0.0018397686071693897, -0.008203108794987202]


In [10]:
batch_size = 10
embedded_chunks = []
for i in range(0, len(chunks), batch_size):
    result = vo.embed(chunks[i:i+batch_size], model="voyage-4")
    for text, vector in zip(chunks[i:i+batch_size], result.embeddings):
        embedded_chunks.append({'text':text, 'vector':vector})
print(len(embedded_chunks))
print(embedded_chunks[0]['text'][:250])


495
tsla-20251231 UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 FORM 10-K (Mark One) x ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 For the fiscal year ended December 31 , 2025 OR o TRANSI


In [11]:
import chromadb
client = chromadb.Client()
collection = client.create_collection(name = "tesla_10k")

collection.add(
    ids=[str(i) for i in range(len(embedded_chunks))],
    embeddings=[chunk['vector'] for chunk in embedded_chunks],
    documents=[chunk['text'] for chunk in embedded_chunks]
)
print(collection.count())


495


In [12]:
query_result = collection.query(
    query_embeddings=[vo.embed(["What was Tesla's total revenue?"], model="voyage-4").embeddings[0]],
    n_results=3
)
print(query_result['documents'])

[[' 92,614 95,341 94,133 Automotive leasing 1,712 1,827 2,120 Energy generation and storage leasing 501 522 520 Total revenues $ 94,827 $ 97,690 $ 96,773 Automotive Segment Automotive Sales Automotive sales revenue includes revenues related to cash and financing deliveries of new vehicles, and specific other features and services that meet the definition of a performance obligation under ASC 606, Revenue from Contracts with Customers (“ASC 606”), including internet connectivity, access to our FSD (Supervised) features and their ongoing maintenance, free Supercharging programs and over-the-air software updates. We recognize revenue on automotive sales, net of any discounts or financial subsidies, upon delivery to the customer, which is when the control of a vehicle transfers. Payments are typically received at the point control transfers or in accordance with payment terms customary to the business, except sales we finance for which payments are collected over the contractual loan term.

In [ ]:
def generate_answer(question, retrieved_chunks):
    import anthropic
    from dotenv import load_dotenv
    load_dotenv()
    client = anthropic.Anthropic()
    prompt = f"""Answer the following question based on the provided documents
    Documents: {retrieved_chunks}
    Question: {question}"""
    